<a href="https://colab.research.google.com/github/lyzee0/FileShare/blob/main/ChannelExit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:

# @title 📡 Telegram Mass Channel Exit Bot - Google Colab (FIXED AUTH)
# @markdown Complete working version with proper authentication

# Install required packages
!pip install telethon -q
!pip install nest-asyncio -q

import asyncio
import nest_asyncio
import sys
import os
from telethon import TelegramClient, errors
from telethon.tl.functions.channels import LeaveChannelRequest
from telethon.sessions import StringSession
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import time
import logging
import traceback

# Apply nest_asyncio for Colab
nest_asyncio.apply()

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ============================================
# CONFIGURATION - FILL THESE IN
# ============================================

# @title 🔑 API Configuration { display-mode: "form" }

# @markdown Get your API credentials from https://my.telegram.org
API_ID = 27743731  # @param {type:"integer"}
API_HASH = "f33b0ab78cbec9084e3668df0b3330ff"  # @param {type:"string"}

# @markdown Your phone number with country code (e.g., +1234567890)
PHONE_NUMBER = "+917381435215"  # @param {type:"string"}

# @markdown ### Optional Settings
# @markdown Channels to exclude (comma-separated names or IDs)
EXCLUDE_CHANNELS = ""  # @param {type:"string"}

# @markdown Leave delay between channels (seconds)
LEAVE_DELAY = 2.0  # @param {type:"number"}

# @markdown Auto-confirm (skip confirmation prompt)
AUTO_CONFIRM = True  # @param {type:"boolean"}

# @markdown Show session string at the end
SHOW_SESSION = True  # @param {type:"boolean"}

# @markdown ### Enter Verification Code Below
# @markdown Get the code from your Telegram app
VERIFICATION_CODE = ""  # @param {type:"string"}

# @markdown ### 2FA Password (if enabled)
TWO_FA_PASSWORD = "9668"  # @param {type:"password"}

# @markdown ### Use existing session string (optional)
USE_SESSION_STRING = True  # @param {type:"boolean"}
SESSION_STRING = "1BVtsOGcBu3EaNYHKYpZI_T2p2BPsEibmu9qgqpaKaXgf8UL5JF31HCunMzRMdmHYrWN8f0jZiWjsSyTCO6jFor-pAgXMP3TqLqq9Tq42zQP0y0fJcCfC3kRIAs_Faip6xCJifxJ8JzQeDBBLqKBcKH2MeSTS7TP6Y9qaC5qrVVT69zlcSSUXnIFQsfvPxF3jpuorELqxSYQa4xo9GyAsGMIrNrqmXCPC6m17kqPXR7H_vi-v20lpmwrYvS24UVF6L_DbiXhkrfoMkYpvsC52sIiIP3jQ3prk_GZotLTkr7oxSysWX84FwGHdyasrv2sGBBC85-X__V9b2ohAJMHQGxXlcY0Ttf8="  # @param {type:"string"}

# Global variables
client = None
session_string = ""
channels = []
success_count = 0
fail_count = 0
processed_count = 0
is_running = False
auth_done = False

# ============================================
# UI Setup
# ============================================

# Status output widget
status_output = widgets.Output()
progress_bar = widgets.IntProgress(value=0, min=0, max=100, description='Progress:')
progress_bar.style.bar_color = '#51cf66'
progress_bar.layout.width = '100%'

# Stats display
stats_html = widgets.HTML(value="""
<div style="display: flex; justify-content: space-around; padding: 10px; background: #f0f0f0; border-radius: 10px; margin: 10px 0;">
    <div style="text-align: center;">
        <div style="font-size: 24px; font-weight: bold; color: #667eea;" id="total">0</div>
        <div style="font-size: 12px; color: #666;">Total</div>
    </div>
    <div style="text-align: center;">
        <div style="font-size: 24px; font-weight: bold; color: #51cf66;" id="success">0</div>
        <div style="font-size: 12px; color: #666;">Left</div>
    </div>
    <div style="text-align: center;">
        <div style="font-size: 24px; font-weight: bold; color: #ff6b6b;" id="failed">0</div>
        <div style="font-size: 12px; color: #666;">Failed</div>
    </div>
</div>
""")

# Channel list output
channels_output = widgets.Output()

# Buttons
start_button = widgets.Button(
    description='🚀 Start',
    button_style='success',
    icon='play',
    layout=widgets.Layout(width='120px')
)

stop_button = widgets.Button(
    description='⏹ Stop',
    button_style='danger',
    icon='stop',
    layout=widgets.Layout(width='120px'),
    disabled=True
)

refresh_button = widgets.Button(
    description='🔄 Refresh',
    button_style='info',
    icon='refresh',
    layout=widgets.Layout(width='120px')
)

clear_button = widgets.Button(
    description='🗑 Clear Log',
    button_style='warning',
    icon='trash',
    layout=widgets.Layout(width='120px')
)

# Auth status display
auth_status = widgets.HTML(value="""
<div style="padding: 10px; background: #fff3cd; border: 1px solid #ffc107; border-radius: 5px; margin: 10px 0;">
    ⚠️ <b>Authentication Required</b><br>
    Fill in your API credentials and phone number above, then click Start.
</div>
""")

button_box = widgets.HBox([start_button, stop_button, refresh_button, clear_button])

# Full UI
ui = widgets.VBox([
    widgets.HTML("""
    <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                padding: 20px; border-radius: 10px; color: white; margin-bottom: 20px;">
        <h1 style="text-align: center; margin: 0;">📡 Telegram Mass Channel Exit</h1>
        <p style="text-align: center; margin: 5px 0 0 0;">Google Colab Version - Fixed Authentication</p>
    </div>
    """),
    auth_status,
    stats_html,
    progress_bar,
    button_box,
    status_output,
    channels_output
])

display(ui)

# ============================================
# Authentication Functions
# ============================================

async def authenticate():
    """Handle authentication properly"""
    global client, session_string, auth_done

    with status_output:
        clear_output(wait=True)
        print("🔑 Starting authentication...")

    try:
        # Use existing session if provided
        if USE_SESSION_STRING and SESSION_STRING:
            print("📱 Using saved session string...")
            client = TelegramClient(StringSession(SESSION_STRING), API_ID, API_HASH)
            await client.start()
            print("✅ Session restored successfully!")
            auth_done = True
            return True

        # Create new client
        client = TelegramClient(StringSession(), API_ID, API_HASH)

        # Send code request
        print(f"📱 Sending verification code to {PHONE_NUMBER}...")

        # Start the authentication process
        if VERIFICATION_CODE:
            print(f"📱 Using provided verification code: {VERIFICATION_CODE}")
            try:
                await client.start(phone=PHONE_NUMBER, code_callback=lambda: VERIFICATION_CODE)
            except errors.rpcerrorlist.PhoneCodeInvalidError:
                print("❌ Invalid verification code!")
                print("Please check the code in your Telegram app and update VERIFICATION_CODE")
                return False
        else:
            print("⚠️  No verification code provided!")
            print("📱 Please check your Telegram app for the code")
            print("✏️  Enter it in the VERIFICATION_CODE field above")
            print("🔄 Then click Start again")
            return False

        # Check for 2FA
        try:
            if TWO_FA_PASSWORD:
                print("🔐 Authenticating with 2FA...")
                await client.sign_in(password=TWO_FA_PASSWORD)
        except errors.rpcerrorlist.SessionPasswordNeededError:
            print("🔐 2FA is enabled! Please enter your password in TWO_FA_PASSWORD field")
            return False

        # Get user info
        me = await client.get_me()
        auth_done = True

        with status_output:
            clear_output(wait=True)
            print("✅ Successfully authenticated!")
            print(f"👤 Name: {me.first_name} {me.last_name or ''}")
            print(f"📱 Phone: {me.phone}")
            print(f"🆔 User ID: {me.id}")

            # Update auth status
            auth_status.value = """
            <div style="padding: 10px; background: #d4edda; border: 1px solid #28a745; border-radius: 5px; margin: 10px 0;">
                ✅ <b>Authenticated!</b><br>
                Logged in successfully. You can now leave channels.
            </div>
            """

        # Get session string
        session_string = client.session.save()

        if SHOW_SESSION:
            with status_output:
                print("\n🔑 SESSION STRING (Save this for future use):")
                print("=" * 60)
                print(session_string)
                print("=" * 60)
                print("\n💡 To use this session in future runs:")
                print("1. Set USE_SESSION_STRING = True")
                print("2. Paste the session string in SESSION_STRING field")
                print("3. No verification code needed next time!")

        return True

    except errors.rpcerrorlist.PhoneCodeInvalidError:
        with status_output:
            clear_output(wait=True)
            print("❌ Invalid verification code!")
            print("📱 Please check the code in your Telegram app")
            print("✏️  Update the VERIFICATION_CODE field and try again")
        return False
    except errors.rpcerrorlist.SessionPasswordNeededError:
        with status_output:
            clear_output(wait=True)
            print("🔐 Two-factor authentication is enabled!")
            print("✏️  Please enter your 2FA password in the TWO_FA_PASSWORD field")
            print("🔄 Then click 'Start' again")
        return False
    except errors.rpcerrorlist.PhoneNumberInvalidError:
        with status_output:
            clear_output(wait=True)
            print("❌ Invalid phone number!")
            print("📱 Please check your phone number format (include country code)")
        return False
    except Exception as e:
        with status_output:
            clear_output(wait=True)
            print(f"❌ Authentication error: {str(e)}")
            traceback.print_exc()
        return False

async def get_channels():
    """Get all channels"""
    global channels

    with status_output:
        clear_output(wait=True)
        print("📂 Fetching channels...")

    try:
        all_channels = []
        excluded = [x.strip().lower() for x in EXCLUDE_CHANNELS.split(',') if x.strip()]

        async for dialog in client.iter_dialogs():
            if dialog.is_channel:
                # Check exclusion
                skip = False
                name_lower = dialog.name.lower()
                channel_id = str(dialog.id)

                for excl in excluded:
                    if excl in name_lower or excl == channel_id:
                        skip = True
                        break

                if not skip:
                    all_channels.append(dialog)

        channels = all_channels

        # Display channels
        with channels_output:
            clear_output(wait=True)
            print(f"📊 Found {len(channels)} channels:")
            print("-" * 50)

            if channels:
                for i, ch in enumerate(channels[:20], 1):
                    members = getattr(ch.entity, 'participants_count', 'N/A')
                    username = getattr(ch.entity, 'username', '')
                    username_display = f"@{username}" if username else ""
                    print(f"{i:3}. {ch.name[:35]:35} | Members: {str(members):>5} | {username_display}")

                if len(channels) > 20:
                    print(f"\n... and {len(channels) - 20} more channels")
            else:
                print("No channels found!")

            print("-" * 50)

        return channels

    except Exception as e:
        with status_output:
            clear_output(wait=True)
            print(f"❌ Error fetching channels: {str(e)}")
        return []

async def leave_channel_safe(channel):
    """Leave a channel with error handling"""
    try:
        result = await client(LeaveChannelRequest(channel.entity))
        if result:
            return True, None
        return False, "Unknown error"
    except errors.FloodWaitError as e:
        wait_time = e.seconds + 2
        with status_output:
            print(f"⏳ Rate limited! Waiting {wait_time}s...")
        await asyncio.sleep(wait_time)
        return False, "Rate limited"
    except errors.ChannelPrivateError:
        return False, "Channel is private"
    except errors.rpcerrorlist.UserNotMutualContactError:
        return False, "Not a member"
    except Exception as e:
        return False, str(e)[:50]

async def process_channels():
    """Process all channels"""
    global success_count, fail_count, processed_count, is_running

    if not channels:
        with status_output:
            print("❌ No channels to process!")
        return

    total = len(channels)
    success_count = 0
    fail_count = 0
    processed_count = 0

    progress_bar.max = total
    progress_bar.value = 0

    with status_output:
        clear_output(wait=True)
        print(f"🚀 Starting mass leave - {total} channels")
        print("=" * 50)
        print("Press 'Stop' to interrupt\n")

    start_time = time.time()

    for i, channel in enumerate(channels, 1):
        if not is_running:
            with status_output:
                print("\n⏹ Stopped by user")
            break

        # Update progress
        progress_bar.value = i
        processed_count = i

        # Leave channel
        success, error = await leave_channel_safe(channel)

        if success:
            success_count += 1
            with status_output:
                print(f"✅ [{i}/{total}] Left: {channel.name}")
        else:
            fail_count += 1
            with status_output:
                print(f"❌ [{i}/{total}] Failed: {channel.name} - {error}")

        # Update stats
        update_stats(total, success_count, fail_count, i)

        # Delay
        if i < total and is_running:
            await asyncio.sleep(LEAVE_DELAY)

    # Final summary
    elapsed = time.time() - start_time
    with status_output:
        clear_output(wait=True)
        print("\n" + "=" * 60)
        print("📊 FINAL SUMMARY")
        print("=" * 60)
        print(f"✅ Successfully left: {success_count} channels")
        print(f"❌ Failed: {fail_count} channels")
        print(f"📊 Total processed: {processed_count} channels")
        print(f"⏱ Time taken: {elapsed:.1f} seconds")
        print("=" * 60)

        if SHOW_SESSION and session_string:
            print("\n🔑 SESSION STRING (Save for future):")
            print("=" * 60)
            print(session_string)
            print("=" * 60)
            print("\n💡 To use this session next time:")
            print("1. Set USE_SESSION_STRING = True")
            print("2. Copy this string to SESSION_STRING field")
            print("3. No verification needed!")

def update_stats(total, success, fail, processed):
    """Update stats display"""
    stats_html.value = f"""
    <div style="display: flex; justify-content: space-around; padding: 10px; background: #f0f0f0; border-radius: 10px; margin: 10px 0;">
        <div style="text-align: center;">
            <div style="font-size: 24px; font-weight: bold; color: #667eea;">{total}</div>
            <div style="font-size: 12px; color: #666;">Total</div>
        </div>
        <div style="text-align: center;">
            <div style="font-size: 24px; font-weight: bold; color: #51cf66;">{success}</div>
            <div style="font-size: 12px; color: #666;">Left</div>
        </div>
        <div style="text-align: center;">
            <div style="font-size: 24px; font-weight: bold; color: #ff6b6b;">{fail}</div>
            <div style="font-size: 12px; color: #666;">Failed</div>
        </div>
        <div style="text-align: center;">
            <div style="font-size: 24px; font-weight: bold; color: #ffd93d;">{int((processed/total)*100)}%</div>
            <div style="font-size: 12px; color: #666;">Progress</div>
        </div>
    </div>
    """

# ============================================
# Button Handlers
# ============================================

async def start_bot_async():
    """Async function to start the bot"""
    global is_running, client, session_string, auth_done

    is_running = True
    start_button.disabled = True
    stop_button.disabled = False

    try:
        # Validate config
        if API_ID == 0 or not API_HASH:
            with status_output:
                clear_output(wait=True)
                print("❌ ERROR: Please configure API_ID and API_HASH!")
                print("Get them from https://my.telegram.org")
            return

        if not PHONE_NUMBER:
            with status_output:
                clear_output(wait=True)
                print("❌ ERROR: Please enter your phone number!")
            return

        # Authenticate
        if not auth_done:
            success = await authenticate()
            if not success:
                with status_output:
                    print("\n⚠️  Authentication failed")
                    print("📱 Please check your credentials and try again")
                return

        # Get channels
        channels_list = await get_channels()
        if not channels_list:
            return

        # Confirm
        if not AUTO_CONFIRM:
            with status_output:
                print("\n⚠️ " + "=" * 50)
                print(f"⚠️  You are about to leave {len(channels_list)} channels!")
                print("⚠️  Type 'yes' to proceed or 'no' to cancel")
                print("=" * 50)

            # Use input() in a separate thread
            confirm = input("\n👉 Enter your choice: ")
            if confirm.lower() != 'yes':
                with status_output:
                    clear_output(wait=True)
                    print("❌ Operation cancelled by user")
                return

        # Process
        await process_channels()

    except Exception as e:
        with status_output:
            clear_output(wait=True)
            print(f"❌ Error: {str(e)}")
            traceback.print_exc()
    finally:
        is_running = False
        start_button.disabled = False
        stop_button.disabled = True

        if client:
            await client.disconnect()
            with status_output:
                print("\n🔌 Disconnected from Telegram")

def start_bot(_):
    """Start button handler"""
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(start_bot_async())
    except Exception as e:
        with status_output:
            print(f"❌ Error: {str(e)}")
    finally:
        loop.close()

def stop_bot(_):
    """Stop button handler"""
    global is_running
    is_running = False
    with status_output:
        print("⏹ Stopping... (will finish current channel)")

def refresh_channels(_):
    """Refresh channels button handler"""
    if not client:
        with status_output:
            clear_output(wait=True)
            print("❌ Please start the bot first!")
        return

    # Create async task
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        loop.run_until_complete(get_channels())
    finally:
        loop.close()

def clear_log(_):
    """Clear log button handler"""
    with status_output:
        clear_output(wait=True)
        print("🧹 Log cleared!")
    with channels_output:
        clear_output(wait=True)

# Connect buttons
start_button.on_click(start_bot)
stop_button.on_click(stop_bot)
refresh_button.on_click(refresh_channels)
clear_button.on_click(clear_log)

# ============================================
# Instructions
# ============================================

print("\n" + "=" * 60)
print("📡 TELEGRAM MASS CHANNEL EXIT BOT - FIXED")
print("=" * 60)
print("\n📌 IMPORTANT - AUTHENTICATION STEPS:")
print("=" * 60)
print("1. Fill in your API_ID and API_HASH")
print("2. Enter your PHONE_NUMBER with country code")
print("3. Click 'Start' button")
print("4. Check your Telegram app for verification code")
print("5. Enter the code in VERIFICATION_CODE field above")
print("6. Click 'Start' again (the bot will use the code)")
print("7. If you have 2FA, enter password in TWO_FA_PASSWORD field")
print("=" * 60)

if USE_SESSION_STRING and SESSION_STRING:
    print("\n✅ Using saved session string - no verification needed!")

# Show current config status
print("\n📊 Configuration Status:")
print(f"• API ID: {'✅' if API_ID != 0 else '❌'}")
print(f"• API Hash: {'✅' if API_HASH else '❌'}")
print(f"• Phone Number: {'✅' if PHONE_NUMBER else '❌'}")
print(f"• Verification Code: {'✅' if VERIFICATION_CODE else '❌ (Enter when prompted)'}")
print(f"• Exclusions: {EXCLUDE_CHANNELS if EXCLUDE_CHANNELS else 'None'}")
print(f"• Delay: {LEAVE_DELAY}s")
print(f"• Auto-Confirm: {'Yes' if AUTO_CONFIRM else 'No'}")

print("\n" + "=" * 60)
print("✅ Ready! Click the 'Start' button above.")


📡 TELEGRAM MASS CHANNEL EXIT BOT - FIXED

📌 IMPORTANT - AUTHENTICATION STEPS:
1. Fill in your API_ID and API_HASH
2. Enter your PHONE_NUMBER with country code
3. Click 'Start' button
4. Check your Telegram app for verification code
5. Enter the code in VERIFICATION_CODE field above
6. Click 'Start' again (the bot will use the code)
7. If you have 2FA, enter password in TWO_FA_PASSWORD field

✅ Using saved session string - no verification needed!

📊 Configuration Status:
• API ID: ✅
• API Hash: ✅
• Phone Number: ✅
• Verification Code: ❌ (Enter when prompted)
• Exclusions: None
• Delay: 2.0s
• Auto-Confirm: Yes

✅ Ready! Click the 'Start' button above.
📱 Using saved session string...
✅ Session restored successfully!
